To run an unsupervised classification on satellite data using Python you need GDAL, Numpy and Sklearn. If you wish to see the data you will also need Matplotlib. Assuming you have the libraries installed, import them at the start.

> **Scikit-learn** is a free software machine learning library for the Python programming language. It features various classification, regression and clustering algorithms including support-vector machines, random forests, gradient boosting, k-means and DBSCAN, and is designed to interoperate with the Python numerical and scientific libraries NumPy and SciPy.
>
> Learn more: https://scikit-learn.org/stable/index.html
>
> k-Means algorithm: https://scikit-learn.org/stable/modules/clustering.html#k-means

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np
from sklearn import cluster
import gdal
from osgeo import gdal_array
import matplotlib.pyplot as plt

Lets start with a single band 

In [4]:
# Tell GDAL to throw Python exceptions, and register all drivers
gdal.UseExceptions()
gdal.AllRegister()

# Read in raster image 
img_ds = gdal.Open('South_coast.tif', gdal.GA_ReadOnly)

band = img_ds.GetRasterBand(2)

img = band.ReadAsArray()
print(img.shape)

X = img.reshape((-1,1))
print(X.shape)

#why can we not do this?
#X1 = img.flatten()
#print(X1.shape)

## clustering with k-Means algorithm
k_means = cluster.KMeans(n_clusters=8)
k_means.fit(X)

X_cluster = k_means.labels_
X_cluster = X_cluster.reshape(img.shape)

print(len(X_cluster))


AttributeError: ignored

Plot the classified image

In [ ]:
%matplotlib inline  

plt.figure(figsize=(20,20))

plt.imshow(X_cluster, cmap="hsv")
#plt.colorbar()
plt.show()

What about using all 13 bands of Sentinel 2?

In [ ]:
# Tell GDAL to throw Python exceptions, and register all drivers
gdal.UseExceptions()
gdal.AllRegister()

# Read in raster image 
img_ds = gdal.Open(directory + '/S2_may_South_coast_clip.tif', gdal.GA_ReadOnly)


img = np.zeros((img_ds.RasterYSize, img_ds.RasterXSize, img_ds.RasterCount),
               gdal_array.GDALTypeCodeToNumericTypeCode(img_ds.GetRasterBand(1).DataType))

for b in range(img.shape[2]):
    img[:, :, b] = img_ds.GetRasterBand(b + 1).ReadAsArray()
    
print(img.shape)

#reshape
X = img.reshape((-1,img.shape[2])) ## in 13 bands X1 = img.reshape((-1,13))
print(X.shape)


Now fit it

In [ ]:
k_means = cluster.KMeans(n_clusters=8)
k_means.fit(X)

X_cluster = k_means.labels_

X_cluster = X_cluster.reshape(img[:, :, 0].shape) # single band


And plot

In [ ]:
%matplotlib inline  

print(X_cluster.shape)

plt.figure(figsize=(20,20))
plt.imshow(X_cluster, cmap="hsv")

plt.show()

Changing the classification is straight forward. In this example choose MiniBatchKMeans

In [ ]:
MB_KMeans = cluster.MiniBatchKMeans(n_clusters=8)
MB_KMeans.fit(X)

X_cluster = MB_KMeans.labels_

X_cluster = X_cluster.reshape(img[:, :, 0].shape)

Plot the result

In [ ]:
plt.figure(figsize=(20,20))
plt.imshow(X_cluster, cmap="hsv")

plt.show()

Finally save the result to a new geotiff

In [ ]:
## write out to tiff
### single band raster. 

ds = gdal.Open(directory + 'S2_may_South_coast_clip.tif')
band = ds.GetRasterBand(2)
arr = band.ReadAsArray()
[cols, rows] = arr.shape

format = "GTiff"
driver = gdal.GetDriverByName(format)


outDataRaster = driver.Create(directory + 'South_coast_kmeans.gtif', rows, cols, 1, gdal.GDT_Byte)
outDataRaster.SetGeoTransform(ds.GetGeoTransform())             ##sets same geotransform as input
outDataRaster.SetProjection(ds.GetProjection())                 ##sets same projection as input


outDataRaster.GetRasterBand(1).WriteArray(X_cluster)

outDataRaster.FlushCache()        ## remove from memory
del outDataRaster           ## delete the data (not the actual geotiff)